# 07 · Entrenar y publicar

**Qué hace:** para cada fuente entrena el modelo final con todo el historial y la configuración congelada, mide la puerta de calidad en las últimas 8 semanas, arma la ficha y publica el artefacto en el repositorio de modelos. Después comprueba que se puede cargar en un entorno limpio, con solo `rc_pronostico`, y que da las mismas predicciones.

**Decisiones que aplica:** D-17 (puerta: el MAE del modelo en las últimas 8 semanas debe ser menor que el del ingenuo estacional; si no, queda `candidato`), D-22 (repositorio con versiones inmutables y puntero `vigente.json` escrito al final), D-16 (skops, SHA-256 verificado antes de cargar, versión de scikit-learn idéntica), D-06 (un modelo por local, con el plato como variable), D-13 (entran los platos con ≥ 8 semanas de historial).

**Puerta (concreción de D-17):** se entrena con objetivos hasta 56 días antes del último dato y se evalúan 8 cortes semanales posteriores, con `h` de 2 a 8 (escenario operativo), frente al ingenuo estacional sobre las mismas filas. Si el MAE del modelo es menor, pasa.

**Necesita:** `salidas/<fuente>/` (01 y 02) y `configuracion/congelada.json` (04). **Deja:** `repositorio/modelos/…` (no versionado) y `resultados/tablas/publicacion_modelos.*`.

In [1]:
# Parámetros
import os, json, subprocess, hashlib
from pathlib import Path
from dotenv import load_dotenv

RAIZ = Path.cwd().parent if Path.cwd().name == "cuadernos" else Path.cwd()
load_dotenv(RAIZ / ".env")
REPOSITORIO = RAIZ / os.environ.get("RUTA_REPOSITORIO", "repositorio")
CONF = RAIZ / "configuracion"
TABLAS = RAIZ / "resultados" / "tablas"; TABLAS.mkdir(parents=True, exist_ok=True)
LOCALES = {"stuttgart": "stuttgart", "aurea": os.environ.get("LOCAL_AUREA", "00000000-0000-4000-8000-000000000010")}
DIAS_VALIDACION = 56          # últimas 8 semanas
H_OP = (2, 8)
print(REPOSITORIO.relative_to(RAIZ), LOCALES)

repositorio {'stuttgart': 'stuttgart', 'aurea': '00000000-0000-4000-8000-000000000010'}


In [2]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from rc_pronostico.variables import LISTA_VARIABLES, CATEGORICAS, VERSION_VARIABLES, codigos_producto, filas_entrenamiento
from rc_pronostico.baselines import baselines
from rc_pronostico.reglas import platos_entrenables
from rc_pronostico.pronosticar import pronosticar, huella_historial
from rc_pronostico.artefacto import cargar, cargar_version, publicar, uuid_v7, ErrorArtefacto, sha256_archivo

congelada = json.loads((CONF / "congelada.json").read_text(encoding="utf-8"))
congelada_sha = hashlib.sha256((CONF / "congelada.json").read_bytes()).hexdigest()
assert congelada["variables"] == LISTA_VARIABLES and congelada["version_variables"] == VERSION_VARIABLES
congelada["hiperparametros"]

{'learning_rate': 0.05,
 'max_iter': 200,
 'max_leaf_nodes': 15,
 'min_samples_leaf': 50}

## 1. Entrenamiento y puerta de calidad
`entrenar` ajusta el P50 y el P90 con la configuración congelada. `barajar=True` mezcla al azar las etiquetas: sirve para fabricar a propósito un candidato malo y comprobar la puerta.

In [3]:
def regresor(loss, quantile=None):
    kw = {"loss": loss, **({"quantile": quantile} if quantile is not None else {})}
    return HistGradientBoostingRegressor(**kw, **congelada["hiperparametros"], categorical_features=CATEGORICAS,
                                         early_stopping=congelada["early_stopping"], random_state=congelada["random_state"])

def entrenar(filas, barajar=False):
    filas = filas[filas.producto >= 0]                 # solo platos con código (≥ 8 semanas de historial)
    y = np.random.default_rng(congelada["random_state"]).permutation(filas.y.to_numpy()) if barajar else filas.y
    return (regresor(**congelada["p50"]).fit(filas[LISTA_VARIABLES], y),
            regresor(**congelada["p90"]).fit(filas[LISTA_VARIABLES], y))

def puerta(historial, cal, todas, barajar=False):
    # D-17: entrenar hasta 56 días antes del último dato y comparar con el ingenuo en las 8 semanas siguientes
    hasta = historial.fecha.max(); fin_ent = hasta - pd.Timedelta(days=DIAS_VALIDACION)
    cortes = [fin_ent + pd.Timedelta(days=7 * k) for k in range(8)]
    p50, _ = entrenar(todas[todas.fecha_objetivo <= fin_ent], barajar)
    val = todas[todas.fecha_corte.isin(cortes) & (todas.fecha_objetivo > fin_ent) & todas.h.between(*H_OP) & (todas.producto >= 0)]
    pred = np.clip(p50.predict(val[LISTA_VARIABLES]), 0, None)
    ing = baselines(historial, cal, val).ingenuo
    mae_m, mae_i = float(np.abs(val.y - pred).mean()), float(np.abs(val.y - ing).mean())
    return {"tramo": [str((fin_ent + pd.Timedelta(days=1)).date()), str(hasta.date())], "cortes": len(cortes), "filas": int(len(val)),
            "mae_modelo": mae_m, "mae_ingenuo": mae_i, "aprobada": bool(mae_m < mae_i), "criterio": "mae_modelo < mae_ingenuo (h 2 a 8, últimas 8 semanas)"}

## 2. Preparar cada fuente
Platos con ≥ 56 días de historial y alguna venta reciben código de categoría (D-13); el resto va al baseline.

In [4]:
F = {}
for fuente in LOCALES:
    d = RAIZ / "salidas" / fuente
    hist = pd.read_parquet(d / "historial.parquet"); calj = json.loads((d / "calendario.json").read_text(encoding="utf-8"))
    cal = {"dias_cerrados": calj["dias_cerrados"], "feriados": calj["feriados"]}
    ok = platos_entrenables(hist)
    codigos = codigos_producto(hist[hist.producto_id.isin(ok)])
    todas = filas_entrenamiento(hist, cal, hist.fecha.max(), {p: codigos.get(p, -1) for p in hist.producto_id.unique()})
    F[fuente] = dict(hist=hist, cal=cal, codigos=codigos, todas=todas)
    print(f"{fuente}: {hist.producto_id.nunique()} platos, {len(codigos)} con modelo, {len(todas)} filas, hasta {hist.fecha.max().date()}")

stuttgart: 7 platos, 7 con modelo, 40600 filas, hasta 2015-11-07


aurea: 20 platos, 20 con modelo, 110560 filas, hasta 2026-11-17


## 3. Entrenar, medir la puerta y publicar
Una versión nueva por fuente (UUID v7). El puntero `vigente.json` se mueve solo si pasó la puerta.

In [5]:
PUBLICADO = {}
for fuente, x in F.items():
    hist, cal, codigos, todas = x["hist"], x["cal"], x["codigos"], x["todas"]
    val = puerta(hist, cal, todas)
    p50, p90 = entrenar(todas)
    ficha = {
        "modelo_version": uuid_v7(), "algoritmo": "HistGradientBoostingRegressor, cuantiles 0,5 y 0,9", "perdida_p50": congelada["p50"],
        "hiperparametros": congelada["hiperparametros"], "random_state": congelada["random_state"], "configuracion_sha256": congelada_sha,
        "datos_desde": str(hist.fecha.min().date()), "datos_hasta": str(hist.fecha.max().date()), "huella_datos": huella_historial(hist),
        "filas_entrenamiento": int(len(todas[todas.producto >= 0])), "platos_con_modelo": len(codigos), "metricas_validacion": val}
    r = publicar(REPOSITORIO, LOCALES[fuente], p50, p90, codigos, ficha, aprobado=val["aprobada"])
    PUBLICADO[fuente] = {**r, "validacion": val}
    print(f"{fuente}: MAE modelo {val['mae_modelo']:.3f} vs ingenuo {val['mae_ingenuo']:.3f} -> {r['estado']} ({r['modelo_version']})")

stuttgart: MAE modelo 5.234 vs ingenuo 6.221 -> vigente (01a1fa86-0731-7aee-a5b8-0962b963d0cd)


aurea: MAE modelo 4.870 vs ingenuo 6.368 -> vigente (01a1fa86-1bfe-7f37-96c2-2c7b5123d27c)


In [6]:
pd.DataFrame({f: {"versión": v["modelo_version"], "estado": v["estado"], "MAE modelo": round(v["validacion"]["mae_modelo"], 3),
                  "MAE ingenuo": round(v["validacion"]["mae_ingenuo"], 3), "tramo": " a ".join(v["validacion"]["tramo"]),
                  "filas": v["validacion"]["filas"]} for f, v in PUBLICADO.items()}).T

,versión,estado,MAE modelo,MAE ingenuo,tramo,filas
stuttgart,01a1fa86-0731-7aee-a5b8-0962b963d0cd,vigente,5.234,6.221,2015-09-13 a 2015-11-07,385
aurea,01a1fa86-1bfe-7f37-96c2-2c7b5123d27c,vigente,4.87,6.368,2026-09-23 a 2026-11-17,1100


## 4. Un candidato que no pasa la puerta
Se publica a propósito un modelo malo (etiquetas barajadas) para ÁUREA. La puerta debe rechazarlo: queda en el repositorio como `candidato` y el puntero no se mueve.

In [7]:
x = F["aurea"]; local = LOCALES["aurea"]
puntero = REPOSITORIO / "modelos" / local / "vigente.json"
antes_bytes = puntero.read_bytes(); antes_version = json.loads(antes_bytes)["modelo_version"]

val_malo = puerta(x["hist"], x["cal"], x["todas"], barajar=True)
p50_m, p90_m = entrenar(x["todas"], barajar=True)
ficha_m = {"modelo_version": uuid_v7(), "algoritmo": "SABOTAJE: etiquetas barajadas", "huella_datos": huella_historial(x["hist"]),
           "datos_hasta": str(x["hist"].fecha.max().date()), "metricas_validacion": val_malo}
malo = publicar(REPOSITORIO, local, p50_m, p90_m, x["codigos"], ficha_m, aprobado=val_malo["aprobada"])
despues_bytes = puntero.read_bytes()
print(f"candidato: MAE {val_malo['mae_modelo']:.3f} vs ingenuo {val_malo['mae_ingenuo']:.3f} -> {malo['estado']}")
print("puntero sin cambios:", antes_bytes == despues_bytes, "| vigente sigue siendo:", json.loads(despues_bytes)["modelo_version"] == antes_version)
print("el candidato existe en el repositorio:", (REPOSITORIO / "modelos" / local / malo["modelo_version"] / "modelo.skops").exists())

candidato: MAE 10.058 vs ingenuo 6.368 -> candidato
puntero sin cambios: True | vigente sigue siendo: True
el candidato existe en el repositorio: True


In [8]:
# Una versión publicada no se sobrescribe
try:
    publicar(REPOSITORIO, local, p50_m, p90_m, x["codigos"], {**ficha_m}, aprobado=True)
    rechazada = False
except ErrorArtefacto as e:
    rechazada = True; print("rechazada:", e)

rechazada: La versión 01a1fa86-2e84-7f4a-abee-d52aff2c93ce ya existe y no se sobrescribe


## 5. Carga en un entorno limpio
`uv run --isolated --no-project --with ./rc_pronostico` crea un entorno nuevo con **solo** `rc_pronostico` y sus dependencias (sin Jupyter, `pyarrow`, `matplotlib` ni el código de este experimento). Un script corto lee `vigente.json`, verifica la huella y pronostica 7 días de ÁUREA. Se compara con lo que da el cuaderno.

In [9]:
SCRIPT = '''
import sys, json, importlib.util, platform
import numpy as np, pandas as pd, sklearn
from rc_pronostico.artefacto import cargar
from rc_pronostico.pronosticar import pronosticar

raiz, local, hist_csv, cal_json, pet_json, salida = sys.argv[1:7]
historial = pd.read_csv(hist_csv, parse_dates=["fecha"])
calendario = json.load(open(cal_json, encoding="utf-8")); peticion = json.load(open(pet_json, encoding="utf-8"))
artefacto = cargar(raiz, local)                      # verifica SHA-256, scikit-learn y variables
r = pronosticar(historial, calendario, peticion, artefacto)
r.predicciones.to_csv(salida, index=False)
print("RESULTADO " + json.dumps({"modelo_version": r.modelo_version, "huella_datos": r.huella_datos,
      "sha256_modelo": artefacto.ficha["sha256_modelo"], "python": platform.python_version(), "scikit_learn": sklearn.__version__,
      "numpy": np.__version__, "pandas": pd.__version__,
      "ausentes": [m for m in ("pyarrow", "matplotlib", "jupyterlab", "holidays", "httpx") if importlib.util.find_spec(m) is None]}))
'''
SALIDA = RAIZ / "salidas" / "aislada"; SALIDA.mkdir(parents=True, exist_ok=True)
(SALIDA / "carga_aislada.py").write_text(SCRIPT, encoding="utf-8")

h = F["aurea"]["hist"]
h.to_csv(SALIDA / "historial.csv", index=False)
(SALIDA / "calendario.json").write_text(json.dumps(F["aurea"]["cal"]), encoding="utf-8")
productos = sorted(h.producto_id.unique()) + ["producto-que-no-existe"]
peticion = {"fecha_inicio": str((h.fecha.max() + pd.Timedelta(days=1)).date()), "horizonte_dias": 7, "productos": productos}
(SALIDA / "peticion.json").write_text(json.dumps(peticion), encoding="utf-8")

cmd = ["uv", "run", "--isolated", "--no-project", "--with", "./rc_pronostico", "python", str(SALIDA / "carga_aislada.py"),
       str(REPOSITORIO), LOCALES["aurea"], str(SALIDA / "historial.csv"), str(SALIDA / "calendario.json"),
       str(SALIDA / "peticion.json"), str(SALIDA / "predicciones_aislado.csv")]
ejecucion = subprocess.run(cmd, cwd=RAIZ, capture_output=True, encoding="utf-8")
assert ejecucion.returncode == 0, ejecucion.stderr[-1500:]
aislado = json.loads([l for l in ejecucion.stdout.splitlines() if l.startswith("RESULTADO ")][-1][len("RESULTADO "):])
aislado

{'modelo_version': '01a1fa86-1bfe-7f37-96c2-2c7b5123d27c',
 'huella_datos': 'e4c8c83939605de432cce8bdee5aa313265ea95e1c7e1e29e2f8bcfbef738f68',
 'sha256_modelo': 'd015da4cbdaec2574a0cf5e73f815a7db93a7e5a76931926e72a496993eda173',
 'python': '3.12.10',
 'scikit_learn': '1.7.2',
 'numpy': '2.5.5',
 'pandas': '3.1.1',
 'ausentes': ['pyarrow', 'matplotlib', 'jupyterlab', 'holidays', 'httpx']}

In [10]:
# Lo que da el cuaderno con el mismo artefacto cargado del repositorio
artefacto = cargar(REPOSITORIO, LOCALES["aurea"])
mia = pronosticar(h, F["aurea"]["cal"], peticion, artefacto)
del_aislado = pd.read_csv(SALIDA / "predicciones_aislado.csv", parse_dates=["fecha"])
sin_nulos = lambda d: d.assign(motivo=d.motivo.astype(object).where(d.motivo.notna(), ""))
a, b = sin_nulos(mia.predicciones), sin_nulos(del_aislado)
pd.testing.assert_frame_equal(a, b)
print("predicciones idénticas:", len(a), "filas")
print(mia.predicciones.metodo.value_counts().to_dict())
mia.predicciones[mia.predicciones.producto_id.isin(productos[:2] + productos[-1:])].head(16)

predicciones idénticas: 147 filas
{'modelo': 140, 'sin_historial': 7}


,producto_id,fecha,p50,p90,metodo,motivo
0,00000000-0000-4000-8000-000000000431,2026-11-18,20.57,31.03,modelo,None
1,00000000-0000-4000-8000-000000000431,2026-11-19,21.25,30.59,modelo,None
2,00000000-0000-4000-8000-000000000431,2026-11-20,30.19,45.09,modelo,None
3,00000000-0000-4000-8000-000000000431,2026-11-21,37.43,46.36,modelo,None
4,00000000-0000-4000-8000-000000000431,2026-11-22,34.42,46.35,modelo,None
5,00000000-0000-4000-8000-000000000431,2026-11-23,16.48,23.31,modelo,None
6,00000000-0000-4000-8000-000000000431,2026-11-24,17.34,23.84,modelo,None
7,00000000-0000-4000-8000-000000000432,2026-11-18,27.57,44.25,modelo,None
8,00000000-0000-4000-8000-000000000432,2026-11-19,31.60,42.96,modelo,None
9,00000000-0000-4000-8000-000000000432,2026-11-20,43.62,59.29,modelo,None


## 6. Resumen para el informe
Solo cifras deterministas (sin versiones UUID ni fechas de publicación).

In [11]:
filas = []
for fuente, v in PUBLICADO.items():
    z = v["validacion"]; filas.append({"fuente": fuente, "tramo_puerta": " a ".join(z["tramo"]), "filas_puerta": z["filas"],
        "MAE_modelo": z["mae_modelo"], "MAE_ingenuo": z["mae_ingenuo"], "aprobada": z["aprobada"], "estado": v["estado"]})
filas.append({"fuente": "aurea (candidato sabotaje)", "tramo_puerta": " a ".join(val_malo["tramo"]), "filas_puerta": val_malo["filas"],
        "MAE_modelo": val_malo["mae_modelo"], "MAE_ingenuo": val_malo["mae_ingenuo"], "aprobada": val_malo["aprobada"], "estado": malo["estado"]})
resumen = pd.DataFrame(filas).round(4)
resumen.to_csv(TABLAS / "publicacion_modelos.csv", index=False)
cab = "| " + " | ".join(resumen.columns) + " |\n|" + "|".join(["---"] * len(resumen.columns)) + "|\n"
(TABLAS / "publicacion_modelos.md").write_text(cab + "\n".join("| " + " | ".join(map(str, r)) + " |" for r in resumen.to_numpy()) + "\n", encoding="utf-8")
resumen

,fuente,tramo_puerta,filas_puerta,MAE_modelo,MAE_ingenuo,aprobada,estado
0,stuttgart,2015-09-13 a 2015-11-07,385,5.2338,6.2208,True,vigente
1,aurea,2026-09-23 a 2026-11-17,1100,4.8698,6.3682,True,vigente
2,aurea (candidato sabotaje),2026-09-23 a 2026-11-17,1100,10.0582,6.3682,False,candidato


## Verificaciones

In [12]:
for fuente, v in PUBLICADO.items():
    ruta = REPOSITORIO / "modelos" / LOCALES[fuente]
    ficha = json.loads((ruta / v["modelo_version"] / "ficha.json").read_text(encoding="utf-8"))
    for campo in ("modelo_version", "local_id", "algoritmo", "hiperparametros", "variables", "version_variables", "versiones",
                  "datos_desde", "datos_hasta", "huella_datos", "metricas_validacion", "sha256_modelo"):
        assert campo in ficha, (fuente, campo)
    assert ficha["sha256_modelo"] == sha256_archivo(ruta / v["modelo_version"] / "modelo.skops") == v["sha256"]
    assert ficha["variables"] == LISTA_VARIABLES and ficha["huella_datos"] == huella_historial(F[fuente]["hist"])
    assert v["estado"] == ("vigente" if v["validacion"]["aprobada"] else "candidato")
    if v["estado"] == "vigente":
        vig = json.loads((ruta / "vigente.json").read_text(encoding="utf-8"))
        assert vig["modelo_version"] == v["modelo_version"] and vig["sha256"] == v["sha256"]
        assert cargar(REPOSITORIO, LOCALES[fuente]).ficha["modelo_version"] == v["modelo_version"]
assert PUBLICADO["aurea"]["estado"] == "vigente", "el modelo de ÁUREA debe pasar la puerta para la demostración"
# La puerta rechaza al candidato malo y el puntero no se mueve
assert not val_malo["aprobada"] and val_malo["mae_modelo"] > val_malo["mae_ingenuo"] and malo["estado"] == "candidato"
assert antes_bytes == despues_bytes and cargar(REPOSITORIO, local).ficha["modelo_version"] == antes_version != malo["modelo_version"]
assert rechazada, "una versión existente debe rechazarse"
# Carga aislada: mismas predicciones, mismo artefacto, entorno sin el experimento
assert aislado["modelo_version"] == mia.modelo_version == PUBLICADO["aurea"]["modelo_version"]
assert aislado["sha256_modelo"] == PUBLICADO["aurea"]["sha256"] and aislado["huella_datos"] == mia.huella_datos
assert {"pyarrow", "matplotlib", "jupyterlab"} <= set(aislado["ausentes"]), aislado["ausentes"]
assert len(a) == len(productos) * 7 and (a[a.producto_id == "producto-que-no-existe"].metodo == "sin_historial").all()
assert (a.p90 >= a.p50).all() and (a.p50 >= 0).all()
print("OK · modelos publicados, puerta verificada y carga aislada idéntica")

OK · modelos publicados, puerta verificada y carga aislada idéntica
